# Tiny ERA5 → Aurora E2E Pipeline (Debug Notebook)

Downloads **1 day** of ERA5, builds a **future 6-hour precipitation target**, writes **Zarr**.

### Fixes (important)
- Always uses `./data/data_raw` and `./data/data_proc`
- Downloads ERA5 as **ZIP** and extracts the correct `.nc` files
  - `instant` → t2m/u10/v10/msl
  - `accum` → tp
- Validates file header before opening
- Installs missing IO deps (`netcdf4`, `h5netcdf`) if needed


In [1]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import xarray as xr
import zarr
import cdsapi
import shutil
import zipfile
import sys

print("Imports OK")

Imports OK


In [2]:
# If you run into: "unrecognized engine 'h5netcdf'" or NetCDF open issues, run this cell once.

!{sys.executable} -m pip install -q netcdf4 h5netcdf
print("Installed netcdf4 + h5netcdf")

Installed netcdf4 + h5netcdf


In [3]:
# --- Paths: ALWAYS keep artifacts under ./data/ ---
MARKERS = ["README.md", "proposal.tex", ".git"]
p = Path.cwd()
while p != p.parent and not any((p / m).exists() for m in MARKERS):
    p = p.parent
REPO_ROOT = p
os.chdir(REPO_ROOT)

DATA_DIR = REPO_ROOT / "data"
DATA_RAW = DATA_DIR / "data_raw"
DATA_PROC = DATA_DIR / "data_proc"
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROC.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("CWD:", Path.cwd())
print("DATA_RAW:", DATA_RAW)
print("DATA_PROC:", DATA_PROC)

Repo root: c:\Users\Michel\OneDrive\Projects\enm5310-project
CWD: c:\Users\Michel\OneDrive\Projects\enm5310-project
DATA_RAW: c:\Users\Michel\OneDrive\Projects\enm5310-project\data\data_raw
DATA_PROC: c:\Users\Michel\OneDrive\Projects\enm5310-project\data\data_proc


In [4]:
# --- Configuration ---
DATE = "2024-01-01"
HOURS = [f"{h:02d}:00" for h in range(24)]

# Tiny levels for speed
TINY_LEVELS = ["1000", "850", "500", "200"]

# ZIP targets (CDS may return multi-member ZIPs)
single_zip = DATA_RAW / "single.zip"
pl_zip = DATA_RAW / "pl.zip"

# Extracted NetCDFs
single_nc = DATA_RAW / "single_instant.nc"   # t2m,u10,v10,msl
tp_nc = DATA_RAW / "single_accum.nc"         # tp
pl_nc = DATA_RAW / "pl.nc"                   # t,u,v,z,q

print("DATE:", DATE)
print("single_zip:", single_zip)
print("pl_zip:", pl_zip)
print("single_nc:", single_nc)
print("tp_nc:", tp_nc)
print("pl_nc:", pl_nc)

DATE: 2024-01-01
single_zip: c:\Users\Michel\OneDrive\Projects\enm5310-project\data\data_raw\single.zip
pl_zip: c:\Users\Michel\OneDrive\Projects\enm5310-project\data\data_raw\pl.zip
single_nc: c:\Users\Michel\OneDrive\Projects\enm5310-project\data\data_raw\single_instant.nc
tp_nc: c:\Users\Michel\OneDrive\Projects\enm5310-project\data\data_raw\single_accum.nc
pl_nc: c:\Users\Michel\OneDrive\Projects\enm5310-project\data\data_raw\pl.nc


In [5]:
# ERA5 variable definitions (CDS names)
SINGLE_LEVEL_VARS = [
    "2m_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "mean_sea_level_pressure",
    "total_precipitation",
]

PRESSURE_LEVEL_VARS = [
    "temperature",
    "u_component_of_wind",
    "v_component_of_wind",
    "geopotential",
    "specific_humidity",
]

print("Single-level vars:", SINGLE_LEVEL_VARS)
print("Pressure-level vars:", PRESSURE_LEVEL_VARS)
print("Tiny pressure levels:", TINY_LEVELS)

Single-level vars: ['2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind', 'mean_sea_level_pressure', 'total_precipitation']
Pressure-level vars: ['temperature', 'u_component_of_wind', 'v_component_of_wind', 'geopotential', 'specific_humidity']
Tiny pressure levels: ['1000', '850', '500', '200']


In [6]:
# --- Helpers ---
def ymd(date):
    return date.split("-")

def file_head(path: Path, n=8) -> bytes:
    with open(path, "rb") as f:
        return f.read(n)

def looks_like_netcdf(path: Path) -> bool:
    if not path.exists():
        return False
    head = file_head(path, 8)
    return head.startswith(b"CDF") or head.startswith(b"\x89HDF\r\n\x1a\n")

def extract_nc_from_zip(zip_path: Path, out_nc: Path, prefer: str | None = None):
    """Extract exactly one .nc from a CDS zip.
    If prefer is provided, pick the member whose name contains that substring.
    """
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = [n for n in zf.namelist() if n.lower().endswith(".nc")]
        if not members:
            raise RuntimeError(f"No .nc found inside {zip_path}. Members: {zf.namelist()[:10]}")

        if prefer is not None:
            preferred = [n for n in members if prefer.lower() in n.lower()]
            if not preferred:
                raise RuntimeError(f"Could not find a .nc containing '{prefer}' in {members}")
            member = preferred[0]
        else:
            member = members[0]

        zf.extract(member, path=zip_path.parent)
        extracted = zip_path.parent / member

        out_nc.parent.mkdir(parents=True, exist_ok=True)
        if out_nc.exists():
            out_nc.unlink()
        extracted.replace(out_nc)

def ensure_clean(path: Path):
    if path.exists():
        path.unlink()

print("Helpers ready")

Helpers ready


In [7]:
# Download ERA5 (ZIP) + extract correct members
y, m, d = ymd(DATE)
c = cdsapi.Client()

# SINGLE-LEVEL: may contain both instant + accum
need_single = (not single_nc.exists()) or (not looks_like_netcdf(single_nc))
need_tp = (not tp_nc.exists()) or (not looks_like_netcdf(tp_nc))

if need_single or need_tp:
    print("Downloading single-level ERA5 as ZIP...")
    if single_zip.exists():
        single_zip.unlink()
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "format": "netcdf",
            "variable": SINGLE_LEVEL_VARS,
            "year": y,
            "month": m,
            "day": d,
            "time": HOURS,
        },
        str(single_zip),
    )

    # Always re-extract both to avoid mismatched versions
    if single_nc.exists():
        single_nc.unlink()
    if tp_nc.exists():
        tp_nc.unlink()

    print("Extracting instant fields → single_instant.nc...")
    extract_nc_from_zip(single_zip, single_nc, prefer="instant")
    print("Extracting accum fields (tp) → single_accum.nc...")
    extract_nc_from_zip(single_zip, tp_nc, prefer="accum")
else:
    print("single_instant.nc and single_accum.nc already exist and look OK")

# PRESSURE-LEVELS: often a single .nc, but we still handle zip
need_pl = (not pl_nc.exists()) or (not looks_like_netcdf(pl_nc))
if need_pl:
    print("Downloading pressure-level ERA5 as ZIP...")
    if pl_zip.exists():
        pl_zip.unlink()
    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": "reanalysis",
            "format": "netcdf",
            "variable": PRESSURE_LEVEL_VARS,
            "pressure_level": TINY_LEVELS,
            "year": y,
            "month": m,
            "day": d,
            "time": HOURS,
        },
        str(pl_zip),
    )

    if pl_nc.exists():
        pl_nc.unlink()
    print("Extracting pressure-level fields → pl.nc...")
    extract_nc_from_zip(pl_zip, pl_nc, prefer=None)
else:
    print("pl.nc already exists and looks OK")

print("single_instant head:", file_head(single_nc, 8))
print("single_accum head:", file_head(tp_nc, 8))
print("pl head:", file_head(pl_nc, 8))
print("single_instant size MB:", single_nc.stat().st_size / 1024**2)
print("single_accum size MB:", tp_nc.stat().st_size / 1024**2)
print("pl size MB:", pl_nc.stat().st_size / 1024**2)
print("Download+extract complete")

2025-12-16 14:45:56,179 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.


2025-12-16 14:45:56,678 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2025-12-16 14:45:56,679 INFO Request ID is 71a48200-9082-48f5-abad-ddc381f0a046
2025-12-16 14:45:56,817 INFO status has been updated to accepted
2025-12-16 14:46:10,699 INFO status has been updated to running
2025-12-16 14:47:51,590 INFO status has been updated to successful


bf92e20a8eb4938cd8c6e926368281a7.zip:   0%|          | 0.00/178M [00:00<?, ?B/s]

Extracting instant fields → single_instant.nc...
Extracting accum fields (tp) → single_accum.nc...
pl.nc already exists and looks OK
single_instant head: b'\x89HDF\r\n\x1a\n'
single_accum head: b'\x89HDF\r\n\x1a\n'
pl head: b'\x89HDF\r\n\x1a\n'
single_instant size MB: 157.78326320648193
single_accum size MB: 19.874464988708496
pl size MB: 760.7864618301392
Download+extract complete


In [8]:
# Open datasets (explicit engine) + standardize coords
def open_netcdf(path: Path) -> xr.Dataset:
    return xr.open_dataset(path, engine="netcdf4")

def standardize_coords(ds: xr.Dataset) -> xr.Dataset:
    if "valid_time" in ds.coords and "time" not in ds.coords:
        ds = ds.rename({"valid_time": "time"})

    if "longitude" in ds.coords:
        lon = ds.longitude.values
        if np.nanmin(lon) < 0:
            ds = ds.assign_coords(longitude=(lon + 360) % 360)
        ds = ds.sortby("longitude")

    if "latitude" in ds.coords:
        if ds.latitude[0] < ds.latitude[-1]:
            ds = ds.sortby("latitude", ascending=False)
    return ds

ds_instant = standardize_coords(open_netcdf(single_nc))
ds_accum = standardize_coords(open_netcdf(tp_nc))
ds_pl = standardize_coords(open_netcdf(pl_nc))

# Merge instant + accum so tp is present with the other single-level vars
ds_single = xr.merge([ds_instant, ds_accum], compat="override")

print("Instant vars:", list(ds_instant.data_vars))
print("Accum vars:", list(ds_accum.data_vars))
print("Merged single vars:", list(ds_single.data_vars))
print("PL vars:", list(ds_pl.data_vars))
print("Timesteps single:", len(ds_single.time))
print("Timesteps pl:", len(ds_pl.time))

Instant vars: ['t2m', 'u10', 'v10', 'msl']
Accum vars: ['tp']
Merged single vars: ['t2m', 'u10', 'v10', 'msl', 'tp']
PL vars: ['t', 'u', 'v', 'z', 'q']
Timesteps single: 24
Timesteps pl: 24


In [9]:
# Rename to Aurora variable names (only if present)
rename_single = {
    "t2m": "2t",
    "u10": "10u",
    "v10": "10v",
    "msl": "msl",
    "tp": "tp",
}
rename_single = {k: v for k, v in rename_single.items() if k in ds_single.data_vars}
ds_single = ds_single.rename(rename_single)

print("Renamed single vars:", list(ds_single.data_vars))
print("PL vars:", list(ds_pl.data_vars))

required_single = ["2t", "10u", "10v", "msl", "tp"]
missing_single = [v for v in required_single if v not in ds_single.data_vars]
print("Missing single:", missing_single)
assert not missing_single, f"Missing single vars: {missing_single}"

Renamed single vars: ['2t', '10u', '10v', 'msl', 'tp']
PL vars: ['t', 'u', 'v', 'z', 'q']
Missing single: []


In [10]:
# Align times and build future tp6h target
common_times = np.intersect1d(ds_single.time.values, ds_pl.time.values)
ds_single = ds_single.sel(time=common_times)
ds_pl = ds_pl.sel(time=common_times)

tp = ds_single["tp"]

# future 6-hour accumulation: sum(tp[t+1]..tp[t+6])
tp6h = sum(tp.shift(time=-k) for k in range(1, 7))
tp6h = tp6h.isel(time=slice(0, -6))
y = np.log1p(tp6h)

print("Target shape:", y.shape)
print("Target time length:", y.sizes["time"])

Target shape: (18, 721, 1440)
Target time length: 18


In [11]:
# Slice inputs to match target length
T = y.sizes["time"]

single_vars = ["2t", "10u", "10v", "msl"]
pl_vars = ["z", "u", "v", "t", "q"]

ds_single_in = ds_single[single_vars].isel(time=slice(0, T))
ds_pl_in = ds_pl[pl_vars].isel(time=slice(0, T))

print("Input time length:", T)
print("Single dims:", ds_single_in.dims)
print("PL dims:", ds_pl_in.dims)

Input time length: 18
Single dims: FrozenMappingWarningOnValuesAccess({'time': 18, 'latitude': 721, 'longitude': 1440})
PL dims: FrozenMappingWarningOnValuesAccess({'time': 18, 'pressure_level': 4, 'latitude': 721, 'longitude': 1440})


In [12]:
# Stack arrays
single = xr.concat([ds_single_in[v] for v in single_vars], dim="var").assign_coords(var=single_vars)
pl = xr.concat([ds_pl_in[v] for v in pl_vars], dim="var").assign_coords(var=pl_vars)

print("single shape:", single.shape)  # (time, var, lat, lon)
print("pl shape:", pl.shape)          # (time, var, level, lat, lon)
print("y shape:", y.shape)

single shape: (4, 18, 721, 1440)
pl shape: (5, 18, 4, 721, 1440)
y shape: (18, 721, 1440)


In [16]:
# [WRITE ZARR OUTPUT v2] Replace your existing "Write Zarr output" cell with this one
# Fixes: Zarr v3 create_array cannot accept BOTH data and shape.

import time
import stat
import os
import shutil
import zarr
import numpy as np

ZARR_PATH = DATA_PROC / "tiny.zarr"

# If a previous Zarr group handle exists, drop it so Windows releases file handles
try:
    del root
except NameError:
    pass

def _on_rm_error(func, path, exc_info):
    # If it's read-only, make it writable then retry
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception:
        raise

if ZARR_PATH.exists():
    # Retry a few times in case OneDrive/antivirus is holding a lock briefly
    for attempt in range(6):
        try:
            shutil.rmtree(ZARR_PATH, onerror=_on_rm_error)
            break
        except PermissionError:
            if attempt == 5:
                raise
            time.sleep(1.0)

root = zarr.open_group(str(ZARR_PATH), mode="w")

x_single = single.values.astype(np.float32)
x_pl = pl.values.astype(np.float32)
y_arr = y.values.astype(np.float32)

root.create_array(
    name="inputs_single",
    data=x_single,
    chunks=(1, x_single.shape[1], x_single.shape[2], x_single.shape[3]),
    overwrite=True,
)

root.create_array(
    name="inputs_pl",
    data=x_pl,
    chunks=(1, x_pl.shape[1], x_pl.shape[2], x_pl.shape[3], x_pl.shape[4]),
    overwrite=True,
)

root.create_array(
    name="target",
    data=y_arr,
    chunks=(1, y_arr.shape[1], y_arr.shape[2]),
    overwrite=True,
)

root.attrs["latitude"] = ds_single.latitude.values.tolist()
root.attrs["longitude"] = ds_single.longitude.values.tolist()
if "pressure_level" in ds_pl.coords:
    root.attrs["pressure_level"] = ds_pl.pressure_level.values.tolist()
elif "level" in ds_pl.coords:
    root.attrs["pressure_level"] = ds_pl.level.values.tolist()

print("✅ Zarr written:", ZARR_PATH)


✅ Zarr written: c:\Users\Michel\OneDrive\Projects\enm5310-project\data\data_proc\tiny.zarr


In [17]:
# Final sanity check
print("inputs_single:", root["inputs_single"].shape)
print("inputs_pl:", root["inputs_pl"].shape)
print("target:", root["target"].shape)
print("target range:", float(np.nanmin(root["target"][:])), float(np.nanmax(root["target"][:])))

inputs_single: (4, 18, 721, 1440)
inputs_pl: (5, 18, 4, 721, 1440)
target: (18, 721, 1440)
target range: 0.0 0.16106659173965454
